# Klasifikasi 6 Kelas dengan Random Forest

Notebook ini mengekstrak 10 fitur Sentinel-2A: B2, B3, B4, B8, B11, B12, NDVI, NDWI, MNDWI, dan NDBI berdasarkan shapefile digitasi untuk 6 kelas (Sawah, Bangunan, Mangrove, Lahan Hijau, Laut, Danau) di Jawa Timur.

In [ ]:
from pathlib import Path
import rasterio
import numpy as np

# Cari GeoTIFF di folder kerja atau subfolder materi.
nama_tif = 'sentinel2A_jatim.tif'
kandidat_tif = [Path(nama_tif), Path('materi') / nama_tif]
TIF = next((path for path in kandidat_tif if path.exists()), kandidat_tif[0])

if not TIF.exists():
    print('GeoTIFF belum ditemukan.')
    print('Lokasi yang diperiksa:')
    for path in kandidat_tif:
        print(f'  - {path.resolve()}')
    print('Sel selesai tanpa error. Tambahkan file GeoTIFF sebelum menjalankan model.')
else:
    with rasterio.open(TIF) as src:
        jumlah_band = src.count
        ukuran_citra = (src.width, src.height)
        crs_citra = src.crs

    print(f'GeoTIFF ditemukan: {TIF.resolve()}')
    print(f'Ukuran citra: {ukuran_citra[0]} x {ukuran_citra[1]} piksel')
    print(f'Jumlah band: {jumlah_band}')
    print(f'CRS: {crs_citra}')

    # Asumsi 10 band seperti pada klasifikasi.ipynb atau Anda bisa menyesuaikannya nanti
    print('Fitur yang diharapkan: B2, B3, B4, B8, B11, B12, NDVI, NDWI, MNDWI, NDBI')
    print('Lanjutkan ke sel Random Forest.')

In [ ]:
import geopandas as gpd
from rasterio.features import rasterize
from rasterio.warp import transform
from shapely.geometry import Polygon, mapping
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score

OUT_KLAS = Path('klasifikasi_6kelas_jatim.tif')
OUT_GAMBAR = Path('segmentasi_6kelas.png')
NAMA_BAND = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'MNDWI', 'NDBI']
BUFFER_M = 5

classes = {
    'sawah': 0,
    'bangunan': 1,
    'mangrove': 2,
    'lahanhijau': 3,
    'laut': 4,
    'danau': 5
}

base_path = Path('c:/Users/ARIEL/OneDrive/Documents/PSD/materi')

def baca_shapefile(cls_name):
    shp_name = "input.shp" if cls_name == "danau" else f"{cls_name}.shp"
    path = base_path / cls_name / shp_name
    if not path.exists():
        return None
    return gpd.read_file(path)

if TIF.exists():
    with rasterio.open(TIF) as src:
        arr = src.read().astype('float32')
        profil = src.profile
        tf, crs = src.transform, src.crs

    n_band, H, W = arr.shape
    valid = np.isfinite(arr).all(axis=0) & (arr[:6].sum(axis=0) > 0)
    print(f'Ukuran citra                : {W} x {H} piksel')
    print(f'Jumlah band/fitur           : {n_band}')
    print(f'CRS                         : {crs}')

    def petak_ke_polygon(gdf):
        hasil = []
        # Pastikan CRS sama dengan TIF
        if gdf.crs != crs:
            gdf = gdf.to_crs(crs)
        
        for idx, row in gdf.iterrows():
            polygon = row.geometry
            if polygon is None or polygon.is_empty or polygon.area == 0:
                continue
            if not polygon.is_valid:
                polygon = polygon.buffer(0)
            
            kecil = polygon.buffer(-BUFFER_M)
            hasil.append(kecil if not kecil.is_empty and kecil.area > 0 else polygon)
        return hasil

    def ambil(gdf, kelas, offset):
        if gdf is None:
            return np.empty((0, n_band), dtype='float32'), np.empty(0, dtype=int), np.empty(0, dtype=int)
        
        polygons = petak_ke_polygon(gdf)
        if not polygons:
            return np.empty((0, n_band), dtype='float32'), np.empty(0, dtype=int), np.empty(0, dtype=int)
        
        shapes = [(mapping(polygon), offset + index + 1) for index, polygon in enumerate(polygons)]
        ids = rasterize(shapes, out_shape=(H, W), transform=tf, fill=0, dtype='int32')
        mask = (ids > 0) & valid
        features = arr[:, mask].T
        groups = ids[mask]
        print(f'Kelas {kelas}: {len(polygons)} petak, {len(np.unique(groups))} petak masuk, {len(features)} piksel')
        return features, np.full(len(features), kelas, dtype=int), groups

    X_all, y_all, g_all = [], [], []
    offset = 0
    for cls, label in classes.items():
        gdf = baca_shapefile(cls)
        if gdf is not None:
            X_c, y_c, g_c = ambil(gdf, label, offset)
            X_all.append(X_c)
            y_all.append(y_c)
            g_all.append(g_c)
            offset += 10000
    
    X = np.vstack(X_all)
    y = np.concatenate(y_all)
    grup = np.concatenate(g_all)

    jumlah_fitur = X.shape[1]
    jumlah_data = X.shape[0]
    jumlah_petak = len(np.unique(grup))
    
    print('\n=== RINGKASAN DATA ===')
    print(f'Jumlah data piksel berlabel  : {jumlah_data}')
    print(f'Jumlah fitur per piksel      : {jumlah_fitur}')
    print(f'Ukuran X                    : {X.shape}')
    print(f'Jumlah petak yang terbaca    : {jumlah_petak}')

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    train_index, test_index = next(splitter.split(X, y, grup))
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    petak_train = len(np.unique(grup[train_index]))
    petak_test = len(np.unique(grup[test_index]))

    print('\n=== UKURAN TRAINING DAN TESTING ===')
    print(f'Data train                   : {len(X_train)} piksel dari {petak_train} petak')
    print(f'Data testing                 : {len(X_test)} piksel dari {petak_test} petak')

    rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    akurasi = accuracy_score(y_test, pred)
    kappa = cohen_kappa_score(y_test, pred)
    matriks = confusion_matrix(y_test, pred, labels=list(classes.values()))

    print('\n=== HASIL RANDOM FOREST ===')
    print(f'Akurasi                     : {akurasi:.4f}')
    print(f'Kappa                       : {kappa:.4f}')
    print('Matriks kebingungan:')
    print(matriks)
    print('\nKepentingan fitur:')
    # Jika fitur TIF asli belum 10, kita sesuaikan print ini:
    peringkat_fitur = sorted(zip(NAMA_BAND[:jumlah_fitur], rf.feature_importances_), key=lambda item: -item[1])
    for nama_fitur, nilai in peringkat_fitur:
        print(f'  {nama_fitur:8s} {nilai:.4f} ({nilai * 100:.2f}%)')

    # Latih ulang untuk prediksi peta raster
    rf.fit(X, y)
    flat = arr.reshape(n_band, -1)
    hasil_flat = np.full(H * W, 255, dtype='uint8')
    valid_index = np.flatnonzero(valid.ravel())
    
    print("\nMulai memprediksi seluruh piksel raster...")
    for start in range(0, len(valid_index), 200000):
        index = valid_index[start:start + 200000]
        hasil_flat[index] = rf.predict(flat[:, index].T).astype('uint8')
    hasil = hasil_flat.reshape(H, W)

    for key in ('blockxsize', 'blockysize', 'tiled', 'interleave'):
        profil.pop(key, None)
    profil.update(count=1, dtype='uint8', nodata=255, compress='lzw')
    with rasterio.open(OUT_KLAS, 'w', **profil) as dst:
        dst.write(hasil, 1)
    print(f'Tersimpan Raster Prediksi: {OUT_KLAS.resolve()}')
else:
    print("GeoTIFF tidak ditemukan, jalankan kode ini setelah TIF tersedia.")

In [ ]:
if TIF.exists():
    luas_px = abs(tf.a * tf.e)
    
    hasil_evaluasi = {
        'Jumlah fitur': jumlah_fitur,
        'Jumlah data piksel berlabel': jumlah_data,
        'Ukuran X': X.shape,
        'Data train': f'{len(X_train)} piksel ({petak_train} petak)',
        'Data testing': f'{len(X_test)} piksel ({petak_test} petak)',
        'Akurasi uji': f'{akurasi:.4f}',
        'Kappa': f'{kappa:.4f}',
    }
    
    print('| Metrik | Nilai |')
    print('| :--- | ---: |')
    for metrik, nilai in hasil_evaluasi.items():
        print(f'| {metrik} | {nilai} |')
    
    print('\n| Kelas hasil klasifikasi | Piksel | Luas (ha) | Persentase |')
    print('| :--- | ---: | ---: | ---: |')
    total_px = (hasil != 255).sum()
    for cls, label in classes.items():
        n_px = int((hasil == label).sum())
        luas_ha = n_px * luas_px / 10000
        persen = 100 * n_px / total_px if total_px > 0 else 0
        print(f'| {cls.capitalize()} | {n_px:,} | {luas_ha:.2f} | {persen:.2f}% |')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

if TIF.exists():
    # Tampilkan RGB (Band 4, 3, 2)
    # Sesuaikan index band RGB jika TIF Anda berbeda urutannya
    rgb = np.clip(np.nan_to_num(arr[[2, 1, 0]]).transpose(1, 2, 0) / 0.3, 0, 1)
    
    # Enam Warna Kelas: Sawah, Bangunan, Mangrove, Lahan Hijau, Laut, Danau
    warna = ['#d95f02', '#e7298a', '#1b9e77', '#66a61e', '#7570b3', '#a6cee3']
    kelas_cmap = ListedColormap(warna)
    kelas_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5], kelas_cmap.N)
    kelas = np.ma.masked_where(hasil == 255, hasil)
    
    fig, ax = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
    ax[0].imshow(rgb)
    ax[0].set_title('Citra Sentinel-2A (RGB)')
    ax[1].imshow(kelas, cmap=kelas_cmap, norm=kelas_norm, interpolation='nearest')
    ax[1].set_title('Segmentasi 6 Kelas LULC')
    ax[2].imshow(rgb)
    ax[2].imshow(kelas, cmap=kelas_cmap, norm=kelas_norm, alpha=0.5, interpolation='nearest')
    ax[2].set_title('Overlay Citra dan Klasifikasi')
    
    for axis in ax:
        axis.axis('off')
    
    labels = list(classes.keys())
    handles = [plt.Rectangle((0, 0), 1, 1, color=warna[i], label=labels[i].capitalize()) for i in range(6)]
    
    fig.legend(handles=handles, loc='lower center', ncol=6, frameon=False)
    fig.savefig(OUT_GAMBAR, dpi=200, bbox_inches='tight')
    print(f'Gambar tersimpan: {OUT_GAMBAR.resolve()}')
    plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

if TIF.exists():
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        pred,
        labels=list(classes.values()),
        display_labels=[k.capitalize() for k in classes.keys()],
        cmap='Blues',
        ax=ax[0],
        colorbar=False,
        xticks_rotation=45
    )
    ax[0].set_title('Confusion Matrix (Petak Uji)')

    urut = np.argsort(rf.feature_importances_)
    ax[1].barh(range(len(urut)), rf.feature_importances_[urut], color='#2ecc71', edgecolor='black')
    ax[1].set_yticks(range(len(urut)))
    ax[1].set_yticklabels([NAMA_BAND[index] for index in urut])
    ax[1].set_xlabel('Nilai Kepentingan')
    ax[1].set_title('Kepentingan Band dan Indeks (6 Kelas)')
    plt.tight_layout()
    plt.show()